# 03 · Combine

This notebook combines the clean tables into one table. Before each merge, the keys are checked with `assert`s, so the notebook stops if a
key is missing, badly formed or duplicated. Rows that are removed here are recorded in `docs/corrections.csv` (through
`data/interim/combine_corrections.csv`, which `04_validate` adds to the file).

In [1]:
from pathlib import Path

import pandas as pd

INTERIM = Path("../data/interim")

In [2]:
shares = pd.read_parquet(INTERIM / "share_prices.parquet")
inflation = pd.read_parquet(INTERIM / "inflation.parquet")
iso = pd.read_parquet(INTERIM / "country_codes.parquet")

print(shares.shape, inflation.shape, iso.shape)

(528, 3) (2871, 5) (249, 2)


## Keep only countries
Both sources mix groups of countries (the euro area, income groups, world regions) in with countries. A code that isn't in the ISO 3166-1
table isn't a country, so we remove it, and record each removal with its reason.

In [3]:
oecd_groups = shares.loc[~shares["country_code"].isin(iso["country_code"]), "country_code"].value_counts().sort_index()
wb_groups = inflation[~inflation["country_code"].isin(iso["country_code"])].groupby("country_code").agg(
    name=("country", "first"), rows=("year", "size")
)

print("OECD areas that are not countries:", oecd_groups.to_dict())
print("World Bank areas that are not countries:", len(wb_groups))
wb_groups.head(10)

OECD areas that are not countries: {'EA20': 11}
World Bank areas that are not countries: 46


,name,rows
country_code,,
AFE,Africa Eastern and Southern,11
AFW,Africa Western and Central,11
ARB,Arab World,11
CEB,Central Europe and the Baltics,11
CHI,Channel Islands,11
CSS,Caribbean small states,11
EAP,East Asia & Pacific (excluding high income),11
EAR,Early-demographic dividend,11
EAS,East Asia & Pacific,11


In [4]:
combine_corrections = [
    {
        "country_code": code,
        "rule": "OECD area must be a country in the ISO 3166-1 table",
        "action": f"{rows} rows removed",
        "reason": f"{code} is not in the ISO 3166-1 table: it is a group of countries, not a country",
    }
    for code, rows in oecd_groups.items()
] + [
    {
        "country_code": code,
        "rule": "World Bank area must be a country in the ISO 3166-1 table",
        "action": f"{row['rows']} rows removed",
        "reason": f"{row['name']} is not in the ISO 3166-1 table (a region, income group or territory without an ISO code)",
    }
    for code, row in wb_groups.iterrows()
]
combine_corrections = pd.DataFrame(combine_corrections, columns=["country_code", "rule", "action", "reason"])
combine_corrections.to_csv(INTERIM / "combine_corrections.csv", index=False)

shares = shares[shares["country_code"].isin(iso["country_code"])]
inflation = inflation[inflation["country_code"].isin(iso["country_code"])]

print(len(combine_corrections), "removals recorded")
print(shares["country_code"].nunique(), "countries with share prices")
print(inflation["country_code"].nunique(), "countries with inflation")

47 removals recorded
47 countries with share prices
215 countries with inflation


## Check the keys
The merges join on `country_code` (and `year`). These checks must pass before we merge: the codes have the same format in every table, each
key appears once on the side that the merge expects to be unique, and every country with share prices has inflation data to join to.

In [5]:
for name, table in {"shares": shares, "inflation": inflation, "iso": iso}.items():
    assert table["country_code"].str.fullmatch(r"[A-Z]{3}").all(), f"badly formed country code in {name}"

assert iso["country_code"].is_unique, "the ISO table must have one row per country"
assert not shares.duplicated(["country_code", "year"]).any(), "share prices: one row per country and year"
assert not inflation.duplicated(["country_code", "year"]).any(), "inflation: one row per country and year"
assert shares["year"].dtype == inflation["year"].dtype, "year must have the same type on both sides"

no_inflation = sorted(set(shares["country_code"]) - set(inflation["country_code"]))
assert not no_inflation, f"countries with share prices but no World Bank inflation at all: {no_inflation}"
print("all key checks passed")

all key checks passed


## Merge

In [6]:
combined = shares.merge(iso, on="country_code", how="left", validate="many_to_one")
combined = combined.merge(inflation, on=["country_code", "year"], how="left", validate="one_to_one")

assert len(combined) == len(shares), "a left join on unique keys must keep every share price row"
assert combined["alpha2"].notna().all(), "every country should have found its ISO row"
print(len(shares), "rows before,", len(combined), "rows after")
combined.head()

517 rows before, 517 rows after


,country_code,year,share_price_index,alpha2,wb_alpha2,country,inflation_pct
0,AUS,2014,98.61095,AU,AU,Australia,2.487929
1,AUS,2015,100.00000,AU,AU,Australia,1.508368
2,AUS,2016,96.06994,AU,AU,Australia,1.276994
3,AUS,2017,105.64780,AU,AU,Australia,1.948643
4,AUS,2018,109.20510,AU,AU,Australia,1.911400


In [7]:
combined = combined[["country_code", "alpha2", "wb_alpha2", "country", "year", "share_price_index", "inflation_pct"]]
combined.isna().sum()

country_code         0
alpha2               0
wb_alpha2            0
country              0
year                 0
share_price_index    0
inflation_pct        0
dtype: int64

In [8]:
combined.to_parquet(INTERIM / "combined.parquet", index=False)
combined.shape

(517, 7)